# CLIPFORGE Engine v0.1
**Cloud-first video-to-clips prototype**

This notebook:
1. Accepts an uploaded video or an authorized YouTube URL.
2. Extracts audio with FFmpeg.
3. Transcribes English speech with Faster-Whisper.
4. Ranks candidate moments using transparent transcript heuristics.
5. Renders up to five vertical clips with burned-in subtitles.

**Important:** Use videos you own or have permission to process. Downloading videos from YouTube may be restricted by YouTube's terms and the creator's rights. Only use URL downloading where you are authorized and it is permitted.

## 0. Runtime setup
In Colab, choose **Runtime → Change runtime type**. Select a GPU if one is offered, but the notebook also has a CPU fallback. Free GPU availability is not guaranteed.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install faster-whisper yt-dlp pandas

## 1. Imports and workspace

In [ ]:
from pathlib import Path
import os, re, json, math, subprocess, textwrap, csv, zipfile
import pandas as pd

WORK = Path("/content/clipforge")
INPUT_DIR = WORK / "input"
OUTPUT_DIR = WORK / "output"
TEMP_DIR = WORK / "temp"

for folder in (INPUT_DIR, OUTPUT_DIR, TEMP_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Workspace ready:", WORK)

## 2. Choose your video source
Set `SOURCE_MODE` to `"upload"` or `"youtube"`.

For YouTube, provide a URL only when you have the rights/permission to process the video and the download is permitted. If downloading is not permitted, use an authorized original file upload instead.

In [ ]:
SOURCE_MODE = "upload"   # change to "youtube" for an authorized URL

video_path = None

if SOURCE_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    uploaded_name = next(iter(uploaded.keys()))
    video_path = INPUT_DIR / Path(uploaded_name).name
    video_path.write_bytes(uploaded[uploaded_name])
    print("Uploaded:", video_path)

elif SOURCE_MODE == "youtube":
    VIDEO_URL = "PASTE_AUTHORIZED_VIDEO_URL_HERE"
    if "PASTE_AUTHORIZED" in VIDEO_URL:
        raise ValueError("Edit VIDEO_URL and paste an authorized URL.")
    template = str(INPUT_DIR / "source.%(ext)s")
    subprocess.run([
        "yt-dlp", "--no-playlist", "-f", "bv*+ba/b",
        "--merge-output-format", "mp4", "-o", template, VIDEO_URL
    ], check=True)
    matches = list(INPUT_DIR.glob("source.*"))
    if not matches:
        raise RuntimeError("Download did not produce a video file.")
    video_path = max(matches, key=lambda p: p.stat().st_size)
else:
    raise ValueError("SOURCE_MODE must be 'upload' or 'youtube'.")

print("Video:", video_path)
print("Size (MB):", round(video_path.stat().st_size / (1024*1024), 2))

## 3. Extract audio
We create a mono 16 kHz WAV file, a convenient format for speech recognition.

In [ ]:
audio_path = TEMP_DIR / "audio_16k.wav"
subprocess.run([
    "ffmpeg", "-y", "-i", str(video_path),
    "-vn", "-ac", "1", "-ar", "16000",
    "-c:a", "pcm_s16le", str(audio_path)
], check=True)
print("Audio extracted:", audio_path)

## 4. Transcribe with Faster-Whisper
The notebook selects GPU acceleration when available and otherwise uses CPU int8. For a first test, use a short video (2–5 minutes) before trying a full livestream.

In [ ]:
import torch
from faster_whisper import WhisperModel

has_gpu = torch.cuda.is_available()
device = "cuda" if has_gpu else "cpu"
compute_type = "float16" if has_gpu else "int8"

# "base" is a practical starting point. Try "small" later if resources allow.
model = WhisperModel("base", device=device, compute_type=compute_type)

segments_iter, info = model.transcribe(
    str(audio_path),
    language="en",
    vad_filter=True,
    word_timestamps=False,
    beam_size=5
)

transcript_segments = []
for seg in segments_iter:
    text = (seg.text or "").strip()
    if text:
        transcript_segments.append({
            "start": float(seg.start),
            "end": float(seg.end),
            "text": text
        })

if not transcript_segments:
    raise RuntimeError("No speech was detected. Check the audio or try another video.")

transcript_path = OUTPUT_DIR / "transcript.json"
transcript_path.write_text(json.dumps(transcript_segments, indent=2), encoding="utf-8")

print("Detected language:", info.language)
print("Speech segments:", len(transcript_segments))
print("Transcript saved:", transcript_path)
print("\nPreview:")
for row in transcript_segments[:8]:
    print(f"[{row['start']:.1f}-{row['end']:.1f}] {row['text']}")

## 5. Find and rank candidate moments
This first version uses explainable heuristics rather than a paid LLM:
- speech amount / transcript density
- engaging language cues
- questions and emphatic punctuation
- a preference for clips with enough context

A high score is only a shortlist signal, not a guarantee that a clip will go viral. Review the candidates before publishing.

In [ ]:
CLIP_LENGTH = 35       # seconds
STRIDE = 10             # seconds between candidate starts
MAX_CANDIDATES = 10
MAX_OUTPUT_CLIPS = 5

ENGAGING_TERMS = {
    "but", "however", "because", "secret", "mistake", "crazy", "insane",
    "unbelievable", "finally", "never", "always", "why", "how", "what",
    "truth", "actually", "problem", "worst", "best", "first", "biggest",
    "wait", "imagine", "impossible", "happened", "forgot", "revealed"
}

duration = float(subprocess.check_output([
    "ffprobe", "-v", "error", "-show_entries", "format=duration",
    "-of", "default=noprint_wrappers=1:nokey=1", str(video_path)
], text=True).strip())

def score_window(start, end, rows):
    overlapping = [
        r for r in rows
        if r["end"] > start and r["start"] < end
    ]
    text = " ".join(r["text"] for r in overlapping).strip()
    words = re.findall(r"\b[\w']+\b", text.lower())
    word_count = len(words)
    if word_count < 18:
        return None

    density = min(word_count / 75.0, 1.0)
    cue_hits = sum(1 for w in words if w in ENGAGING_TERMS)
    cue_score = min(cue_hits / 5.0, 1.0)
    punctuation_score = min((text.count("?") + text.count("!")) / 3.0, 1.0)
    context_score = 1.0 if 35 <= word_count <= 105 else 0.65

    score = 100 * (
        0.45 * density +
        0.30 * cue_score +
        0.15 * punctuation_score +
        0.10 * context_score
    )
    return {
        "start": round(start, 2),
        "end": round(end, 2),
        "score": round(score, 1),
        "word_count": word_count,
        "text": text
    }

candidates = []
last_start = max(0, duration - CLIP_LENGTH)
start = 0.0
while start <= last_start:
    end = min(start + CLIP_LENGTH, duration)
    candidate = score_window(start, end, transcript_segments)
    if candidate:
        candidates.append(candidate)
    start += STRIDE

if not candidates:
    raise RuntimeError("Not enough speech to form candidate clips.")

candidates.sort(key=lambda x: x["score"], reverse=True)

# Greedy non-overlap selection so the shortlist isn't five versions of the same moment.
selected = []
for cand in candidates:
    overlaps = any(
        not (cand["end"] <= chosen["start"] or cand["start"] >= chosen["end"])
        for chosen in selected
    )
    if not overlaps:
        selected.append(cand)
    if len(selected) >= MAX_CANDIDATES:
        break

candidate_df = pd.DataFrame(selected)
candidate_csv = OUTPUT_DIR / "candidate_clips.csv"
candidate_df.to_csv(candidate_csv, index=False)

display(candidate_df[["start", "end", "score", "word_count", "text"]])
print("Candidate list saved:", candidate_csv)

## 6. Render the top five vertical clips with subtitles
This initial renderer uses a centered 9:16 crop. It does not yet track faces or moving subjects; that comes in a later phase. Subtitle timing is segment-level, not word-by-word.

In [ ]:
def srt_timestamp(seconds):
    ms = int(round(seconds * 1000))
    hours = ms // 3600000
    minutes = (ms % 3600000) // 60000
    secs = (ms % 60000) // 1000
    millis = ms % 1000
    return f"{hours:02}:{minutes:02}:{secs:02},{millis:03}"

def make_srt(clip_start, clip_end, out_path):
    lines = []
    idx = 1
    for seg in transcript_segments:
        a = max(seg["start"], clip_start)
        b = min(seg["end"], clip_end)
        if b <= a:
            continue
        # Break long transcript segments into readable subtitle lines.
        caption = textwrap.fill(seg["text"], width=42)
        lines.extend([
            str(idx),
            f"{srt_timestamp(a-clip_start)} --> {srt_timestamp(b-clip_start)}",
            caption,
            ""
        ])
        idx += 1
    out_path.write_text("\n".join(lines), encoding="utf-8")

rendered = []
for i, cand in enumerate(selected[:MAX_OUTPUT_CLIPS], start=1):
    clip_start, clip_end = cand["start"], cand["end"]
    clip_path = OUTPUT_DIR / f"clip_{i:02}.mp4"
    srt_path = TEMP_DIR / f"clip_{i:02}.srt"
    make_srt(clip_start, clip_end, srt_path)

    # Escape the subtitle path for FFmpeg's filter syntax.
    srt_filter_path = str(srt_path).replace("\\", "/").replace(":", "\\:")
    vf = (
        "scale=1080:1920:force_original_aspect_ratio=increase,"
        "crop=1080:1920,"
        f"subtitles='{srt_filter_path}':"
        "force_style='FontName=Arial,FontSize=18,PrimaryColour=&H00FFFFFF,"
        "OutlineColour=&H00000000,BorderStyle=1,Outline=2,Shadow=0,"
        "Alignment=2,MarginV=110'"
    )

    cmd = [
        "ffmpeg", "-y", "-ss", str(clip_start), "-i", str(video_path),
        "-t", str(clip_end-clip_start),
        "-vf", vf,
        "-c:v", "libx264", "-preset", "ultrafast", "-crf", "23",
        "-c:a", "aac", "-b:a", "128k", "-movflags", "+faststart",
        str(clip_path)
    ]
    subprocess.run(cmd, check=True)
    rendered.append(clip_path)
    print("Rendered:", clip_path.name)

print(f"Finished: {len(rendered)} clips")

## 7. Download the results

In [ ]:
from google.colab import files
import shutil

zip_path = WORK / "clipforge_output.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.iterdir():
        if path.is_file():
            zf.write(path, arcname=path.name)

files.download(str(zip_path))

## 8. Phase 1 test checklist
- [ ] A short video uploads successfully.
- [ ] Audio extraction completes.
- [ ] English transcript has sensible timestamps.
- [ ] Candidate moments are not mostly silent or repetitive.
- [ ] Five clips render (or fewer if the source is short).
- [ ] Vertical framing is acceptable.
- [ ] Subtitles are readable and synchronized.

**Known limitations:** transcript-only scoring can miss visual jokes, gameplay highlights, facial reactions and moments where the audio context is misleading. Center cropping can cut off speakers. The first version is an MVP for testing, not a finished commercial service.